In [ ]:
## Task 1: Visualizing the Mechanics
We visualize the Two-Parameter Logistic (2PL) Item Response Theory (IRT) model to examine how item difficulty ($b_i$) and item discrimination ($a_i$) shape the probability of a correct response.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-6, 6, 300)

curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

fig = go.Figure()
for curve in curves:
    a, b, style = curve["a"], curve["b"], curve["line_style"]
    fig.add_trace(go.Scatter(
        x=theta_vals, y=p_i(theta_vals, a, b),
        mode='lines', name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

fig.update_layout(
    title={'text': "Two-Parameter Logistic (2PL) Item Response Curves", 'x': 0.5},
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i=1|θ)",
    template="plotly_white"
)
fig.show()

### Task 1 Interpretation
Moving $b_i$ horizontally shifts the curve along the $\theta$-axis[cite: 1]. The parameter $b_i$ represents the difficulty level where $P(Y_i=1|\theta) = 0.5$[cite: 1]. Increasing $b_i$ moves the curve to the right, requiring higher ability $\theta$ to achieve the same probability of success[cite: 1].

## Task 2: Sequential Likelihood Contribution

* **Single Item Likelihood $L(y_k|\theta)$[cite: 1]:**
  $$L(y_k|\theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$[cite: 1]

* **Joint Likelihood Function $L(y^{(k)}|\theta)$[cite: 1]:**
  Assuming conditional independence given $\Theta=\theta$[cite: 1]:
  $$L(y^{(k)}|\theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$[cite: 1]

## Task 3: Mathematical Formulation of the Running Update

* **Recursive Posterior Density[cite: 1]:**
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto L(y_k|\theta) \cdot f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})$$[cite: 1]

* **Explicit Update with Normalizer[cite: 1]:**
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) = \frac{[p_k(\theta)^{y_k}(1 - p_k(\theta))^{1 - y_k}] f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})}{\int_{\mathbb{R}} [p_k(s)^{y_k}(1 - p_k(s))^{1 - y_k}] f_{\Theta|Y^{(k-1)}}(s|y^{(k-1)}) \, ds}$$[cite: 1]

## Task 4: Dynamic Shifting Mechanics

For a correct response ($y_k = 1$), the likelihood contribution is:
$$L(y_k|\theta) = p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$$

This is a monotonically increasing function of $\theta$. When multiplied by the prior $f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})$, it weights higher values of $\theta$ much more heavily. If $b_k$ is large, $p_k(\theta)$ remains small until higher $\theta$ values, creating a steep positive gradient that significantly pushes the peak (mode) of the updated posterior density to the right.

## Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the steepness of the logistic curve $p_k(\theta)$:
* **Very Large $a_k$:** $p_k(\theta)$ approaches a step function around $b_k$. Multiplying by this sharp threshold heavily penalizes regions on one side of $b_k$, rapidly shrinking the variance (increasing the sharpness/certainty) of the posterior distribution.
* **Very Small $a_k$:** $p_k(\theta)$ becomes nearly flat across $\theta$. The likelihood contribution is almost uninformative, causing minimal change to the posterior variance or peak.

## Task 6: Numerical Implementation of a Running Grid

1. Define a fine discrete grid $\theta \in [\theta_{min}, \theta_{max}]$ (e.g., 500 points from -5 to 5)[cite: 1].
2. Initialize the density array using the standard normal prior PDF $f^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}e^{-\theta^2/2}$[cite: 1].
3. At step $k$, compute $p_k(\theta)$ across grid points and evaluate likelihood $L_k = (p_k)^{y_k}(1-p_k)^{1-y_k}$[cite: 1].
4. Perform element-wise multiplication: $\tilde{f}^{(k)} = f^{(k-1)} \odot L_k$[cite: 1].
5. **Sequential Normalization:** Numerically compute the integral $Z_k = \int \tilde{f}^{(k)}(\theta) d\theta$ using the trapezoidal rule (`np.trapezoid`), and update $f^{(k)} = \frac{\tilde{f}^{(k)}}{Z_k}$[cite: 1].

## Task 7: Evaluating Convergence over the Timeline

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))
current_posterior = stats.norm.pdf(theta_grid, 0, 1)

for k in range(n_items):
    a_k, b_k = a_params[k], b_params[k]
    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    current_posterior = current_posterior * likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    theta_bayes_k = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior)]

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Ability (0.75)")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers', name='MAP Estimate'))
fig2.update_layout(title="Convergence of Latent Ability Estimators Over Time", xaxis_title="Item Position (k)", yaxis_title="Estimated Ability (θ)", template="plotly_white")
fig2.show()

### Task 7 Analysis
As $k$ increases, the distance between both estimators ($\hat{\theta}_{Bayes}$ and $\hat{\theta}_{MAP}$) and $\theta_{true}$ contracts toward zero[cite: 1]. This indicates that each observation adds information, reducing posterior variance and demonstrating that the platform's confidence in its measurement increases over time[cite: 1].

#Section 2

## Task 1: Structural Probability and Properties

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0, 1, 500)
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: Beta(1,1)", "color": "gray"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed: Beta(2,8)", "color": "blue"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed: Beta(8,2)", "color": "green"},
]

fig = go.Figure()
for config in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid, config["alpha"], config["beta"])
    fig.add_trace(go.Scatter(x=theta_grid, y=pdf_vals, mode='lines', name=config["name"]))

fig.update_layout(title="Beta(α, β) PDF Variations", xaxis_title="θ", yaxis_title="Density", template="plotly_white")
fig.show()

### Task 1 Interpretation
* $\alpha = 1, \beta = 1$: Uniform flat distribution over $[0, 1]$[cite: 1].
* $\alpha < \beta$ ($\alpha=2, \beta=8$): Center of mass shifts to the left (right-skewed), placing higher probability on lower conversion rates[cite: 1].
* $\alpha > \beta$ ($\alpha=8, \beta=2$): Center of mass shifts to the right (left-skewed), placing higher probability on higher conversion rates[cite: 1].

## Task 2: Sequential Likelihood and Joint History

* **Single Response Likelihood[cite: 1]:**
  $$L(y_k|\theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$[cite: 1]

* **Joint Likelihood Function[cite: 1]:**
  $$L(y^{(k)}|\theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{C_k} (1 - \theta)^{k - C_k}$$[cite: 1]
  where $C_k = \sum_{i=1}^k y_i$ represents total clicks[cite: 1].

## Task 3: Closed-Form Analytical Updates (Conjugacy)

* **Proof of Beta-Binomial Conjugacy[cite: 1]:**
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto L(y_k|\theta) \cdot f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})$$[cite: 1]
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto [\theta^{y_k}(1-\theta)^{1-y_k}] \cdot [\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}]$$[cite: 1]
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$[cite: 1]

* **Closed-Form Updates[cite: 1]:**
  $$\alpha_k = \alpha_{k-1} + y_k, \quad \beta_k = \beta_{k-1} + (1 - y_k)$$[cite: 1]

* **Posterior Mean Formula[cite: 1]:**
  $$\mathbb{E}[\Theta|Y^{(k)}=y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + C_k}{\alpha_0 + \beta_0 + k}$$[cite: 1]

## Task 4: Dynamic Shifting Mechanics

* **Mechanics:**
  * A click ($y_k = 1$) increments $\alpha_k$ by 1, pushing the peak of the Beta distribution to the right[cite: 1].
  * A non-click ($y_k = 0$) increments $\beta_k$ by 1, pushing the peak to the left[cite: 1].
* **Contrast to Non-Conjugate Models:**
  In conjugate setups like Beta-Binomial, the posterior distribution stays within the same parametric family (Beta), requiring simple addition to update parameters[cite: 1]. Non-conjugate models (like 2PL IRT) cannot be simplified algebraically into a known distribution, forcing numerical integration over a grid[cite: 1].

## Task 5: Running Point Estimators

* **Posterior Mean $(\hat{\theta}_{Bayes}^{(k)})$[cite: 1]:**
  $$\hat{\theta}_{Bayes}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$[cite: 1]

* **Maximum A Posteriori $(\hat{\theta}_{MAP}^{(k)})$[cite: 1]:**
  $$\hat{\theta}_{MAP}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$$[cite: 1]

## Task 6: Performance Tracking and Convergence Analysis

import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
theta_true = 0.35
n_impressions = 100
steps = list(range(n_impressions + 1))

alpha_param, beta_param = 1, 1
running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha_param += y_k
    beta_param += (1 - y_k)

    theta_bayes_k = alpha_param / (alpha_param + beta_param)
    theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2) if (alpha_param > 1 and beta_param > 1) else 0.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True CTR (0.35)")
fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode='lines', name='MAP Estimate'))
fig.update_layout(title="Beta-Binomial Convergence Timeline", xaxis_title="Impressions (k)", yaxis_title="Estimated CTR (θ)", template="plotly_white")
fig.show()

### Task 6 Analysis
As sample size $k \to 100$, the estimators converge tightly to $\theta_{true} = 0.35$[cite: 1]. The influence of the initial prior $(\alpha_0, \beta_0)$ diminishes asymptotically as real-world evidence accumulates[cite: 1].

#section 3

## Task 1: Prior Belief Boundaries

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0.01, 1.0, 500)
prior = stats.beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior, mode='lines', name='Beta(8, 1.5) Prior'))
fig.update_layout(title="Initial Structural Health Prior Distribution", xaxis_title="Stiffness Factor (θ)", yaxis_title="Density", template="plotly_white")
fig.show()

### Task 1 Expected Value & Engineering Rationale
$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} \approx 0.842$$

This left-skewed Beta distribution concentrates most probability density near $1.0$, accurately capturing the initial physical state of a newly manufactured or healthy structural component[cite: 1].

## Task 2: Structural Likelihood Formulation

* **Single Sensor Measurement Likelihood[cite: 1]:**
  Given $y_k = \theta \cdot K_{nominal} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$[cite: 1]:
  $$L(y_k|\theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln y_k - \ln(\theta K_{nominal}))^2}{2\sigma^2} \right)$$[cite: 1]

* **Joint Likelihood Function[cite: 1]:**
  $$L(y^{(k)}|\theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln y_i - \ln(\theta K_{nominal}))^2}{2\sigma^2} \right)$$[cite: 1]

## Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

* **Why No Closed-Form Exists[cite: 1]:**
  Combining a Beta prior over $\theta \in (0, 1]$ with a log-normal measurement likelihood leads to a non-conjugate posterior density function whose denominator integral cannot be evaluated analytically[cite: 1].

* **Recursive Relationship[cite: 1]:**
  $$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto L(y_k|\theta) \cdot f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})$$[cite: 1]

## Task 4: Running Point Estimates

* **Running Posterior Mean $(\hat{\theta}_{Bayes}^{(k)})$[cite: 1]:**
  $$\hat{\theta}_{Bayes}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \, d\theta$$[cite: 1]

* **Running MAP Estimate $(\hat{\theta}_{MAP}^{(k)})$[cite: 1]:**
  $$\hat{\theta}_{MAP}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta|Y^{(k)}}(\theta|y^{(k)})$$[cite: 1]

## Task 5: Algorithmic Grid Approximation and Normalization

1. **Discretization:** Set $M$ grid points over $[0.01, 1.0]$ with spacing $\Delta \theta = \frac{1.0 - 0.01}{M-1}$[cite: 1].
2. **Initialization:** Evaluate prior $P_0(\theta_m)$ over grid and normalize using `np.trapezoid`[cite: 1].
3. **Updating:** Calculate $P_{unnorm}^{(k)}(\theta_m) = P^{(k-1)}(\theta_m) \cdot L(y_k|\theta_m)$[cite: 1].
4. **Normalization:** Compute $Z_k = \text{trapezoid}(P_{unnorm}^{(k)}, \theta)$ and set $P^{(k)} = P_{unnorm}^{(k)} / Z_k$[cite: 1].

## Task 6: Performance Tracking and Degradation Convergence Analysis

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)
theta_true = 0.68
sigma = 0.15
K_nominal = 50.0
n_readings = 15

theta_grid = np.linspace(0.01, 1.0, 500)
current_posterior = stats.beta.pdf(theta_grid, 8, 1.5)
current_posterior /= np.trapezoid(current_posterior, theta_grid)

milestones = [0, 1, 2, 5, 10, 15]
fig = go.Figure()

for k in range(1, n_readings + 1):
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    current_posterior = current_posterior * likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    if k in milestones:
        fig.add_trace(go.Scatter(x=theta_grid, y=current_posterior, mode='lines', name=f"Step {k}"))

fig.add_vline(x=theta_true, line_dash="dot", line_color="red", annotation_text="True Damage (0.68)")
fig.update_layout(title="SHM Bounded Bayesian Parameter Estimation", xaxis_title="Stiffness Factor (θ)", yaxis_title="Probability Density", template="plotly_white")
fig.show()

### Task 6 Analysis
Within 3–5 sensor readings, the likelihood updates overcome the initially optimistic prior ($\mathbb{E} \approx 0.84$) and shift the posterior peak toward $\theta = 0.68$[cite: 1]. Narrowing density curves reflect decreasing uncertainty, allowing engineers to reliably trigger safety thresholds[cite: 1].

#section 4

## Part 1: Deriving the Marginal Density

$$p(x_i) = \sum_{k=1}^K P(X_i=x_i, C_i=k) = \sum_{k=1}^K P(C_i=k) P(X_i=x_i|C_i=k)$$[cite: 1]
$$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i|\mu_k, \Sigma_k)$$[cite: 1]

*Explanation:* It is called a Gaussian mixture density because it represents a complex distribution by linearly combining (mixing) $K$ individual Gaussian component densities weighted by prior proportions $\phi_k$[cite: 1].

## Part 2: Deriving the Posterior Cluster Probability

$$\gamma_{ik} = P(C_i=k|X_i=x_i) = \frac{P(X_i=x_i|C_i=k) P(C_i=k)}{p(x_i)} = \frac{\phi_k \mathcal{N}(x_i|\mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i|\mu_j, \Sigma_j)}$$[cite: 1]

*Interpretation:* $\gamma_{ik}$ is the posterior probability (responsibility) of cluster $k$ after updating the prior belief $\phi_k$ with observation $x_i$[cite: 1].

## Part 3: One-Hot Encoding of Latent Variables

$$\mathbb{E}[Z_{ik}|X_i=x_i] = 1 \cdot P(Z_{ik}=1|X_i=x_i) + 0 \cdot P(Z_{ik}=0|X_i=x_i) = P(C_i=k|X_i=x_i) = \gamma_{ik}$$[cite: 1]
$$\mathbb{E}[Z_i|X_i=x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$[cite: 1]

*Conclusion:* Soft cluster assignment in GMM is precisely the conditional expectation of the one-hot encoded latent variable vector $Z_i$ given $x_i$[cite: 1].

## Part 4: Soft vs. Hard Clustering

* **Soft Clustering ($\mathbb{E}[Z_i|X_i=x_i]$):** Assigns a probability vector in $[0, 1]^K$, preserving uncertainty and boundary overlaps[cite: 1].
* **Hard Clustering ($\hat{C}_i = \arg\max_k \gamma_{ik}$):** Collapses probabilities into a single discrete class label, removing ambiguity[cite: 1].

## Part 5: Expectation of Observation Given Cluster

$$\mathbb{E}[X_i|C_i=k] = \int x_i \mathcal{N}(x_i|\mu_k, \Sigma_k) dx_i = \mu_k$$[cite: 1]

*Comparison:*
* $\mathbb{E}[Z_i|X_i=x_i]$ maps observation space to probability space (soft membership of a point)[cite: 1].
* $\mathbb{E}[X_i|C_i=k]$ maps cluster identity to spatial coordinates (mean location/center of a cluster)[cite: 1].

## Part 6: The Complete-Data Likelihood

$$p(X, Z) = \prod_{i=1}^n \prod_{k=1}^K [\phi_k \mathcal{N}(x_i|\mu_k, \Sigma_k)]^{z_{ik}}$$[cite: 1]
$$l_c = \ln p(X, Z) = \sum_{i=1}^n \sum_{k=1}^K z_{ik} [\ln \phi_k + \ln \mathcal{N}(x_i|\mu_k, \Sigma_k)]$$[cite: 1]

*Why Easy to Maximize:* If $z_{ik}$ were known, terms for each cluster $k$ uncouple completely, turning the problem into $K$ standard independent MLE problems[cite: 1].

## Part 7: The EM Interpretation

$$Q = \mathbb{E}_{Z|X}[l_c] = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} [\ln \phi_k + \ln \mathcal{N}(x_i|\mu_k, \Sigma_k)]$$[cite: 1]

*Interpretation:* The E-step conditionally updates cluster memberships using current parameter estimates via Bayes' Rule[cite: 1].

## Part 8: Parameter Updates (M-step)

$$N_k = \sum_{i=1}^n \gamma_{ik}, \quad \phi_k^{new} = \frac{N_k}{n}$$[cite: 1]
$$\mu_k^{new} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i, \quad \Sigma_k^{new} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{new})(x_i - \mu_k^{new})^T$$[cite: 1]

*Role of Responsibilities:* $\gamma_{ik}$ acts as a fractional weight in calculating updated parameters[cite: 1].

## Part 9: Summary Interpretation

GMM clustering operates as an iterative conditional update process[cite: 1]. Starting with prior weight $\phi_k$, it evaluates likelihood $\mathcal{N}(x_i|\mu_k, \Sigma_k)$ to measure point compatibility[cite: 1]. Bayes' Rule yields posterior responsibility $\gamma_{ik}$, forming the soft assignment vector $\mathbb{E}[Z_i|X_i=x_i]$[cite: 1]. The M-step then re-estimates component parameters using these posterior probabilities as weights[cite: 1].

## Part 10: Computational Simulation and Out-of-Sample Validation

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.model = GaussianMixture(n_components=n_components, covariance_type="full", random_state=random_state)

    def prepare_data(self, df, feature_cols, test_size=0.2):
        X = df[feature_cols].dropna().values
        X_scaled = self.scaler.fit_transform(X)
        return train_test_split(X_scaled, test_size=test_size, random_state=42)

    def fit(self, X_train):
        self.model.fit(X_train)
        print(f"Converged: {self.model.converged_}, Iterations: {self.model.n_iter_}")

    def evaluate(self, X_test):
        score = self.model.score(X_test)
        print(f"Test Average Log-Likelihood: {score:.4f}")
        return score

    def plot_density_heatmap(self, X_train, feature_names):
        X_orig = self.scaler.inverse_transform(X_train)
        fig = px.density_heatmap(x=X_orig[:, 0], y=X_orig[:, 1], labels={"x": feature_names[0], "y": feature_names[1]}, marginal_x="histogram", marginal_y="histogram")
        fig.show()